# Tarea 5: Módulo NLP (Procesamiento de Lenguaje Natural)

Este notebook contiene el procesamiento NLP para extraer features a partir de noticias deportivas de selecciones. Implementamos un corpus de titulares realistas en el periodo del Mundial (2018-2024), extraemos entidades y palabras clave usando SpaCy, y realizamos análisis de sentimiento multilingüe con el modelo preentrenado de HuggingFace `nlptown/bert-base-multilingual-uncased-sentiment`.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import spacy
from transformers import pipeline
import sys

PROCESSED_DIR = "../data/processed"
clean_data_path = os.path.join(PROCESSED_DIR, "matches_clean.csv")

df = pd.read_csv(clean_data_path)
df['date'] = pd.to_datetime(df['date'])
print(f"Cargado dataset limpio de partidos con {len(df)} encuentros.")

# Add dashboard path to python path to import api_helper
sys.path.append("../dashboard")
import api_helper


### 1. Búsqueda de Noticias Reales (APIs en vivo)

En lugar de utilizar un corpus artificial con noticias simuladas, nos conectamos en tiempo real con Google News RSS para buscar titulares deportivos reales asociados con las selecciones.


In [ ]:
teams = ['Argentina', 'Brazil', 'Spain', 'France', 'Germany', 'England', 'Portugal', 'Italy', 'Mexico', 'USA', 
         'Croatia', 'Netherlands', 'Belgium', 'Uruguay', 'Japan', 'Senegal', 'Morocco', 'Saudi Arabia', 'Ecuador', 'Canada']

real_news_records = []
print("Consultando titulares reales de Google News RSS...")
for team in teams:
    headlines = api_helper.fetch_rss_headlines(team, max_results=5)
    print(f"- {team}: obtenidos {len(headlines)} titulares.")
    for hl in headlines:
        real_news_records.append({
            'team': team,
            'headline': hl
        })

news_df = pd.DataFrame(real_news_records)
print(f"\nTotal de noticias reales recopiladas: {len(news_df)}")


### 2. Análisis de Sentimiento de Noticias Reales con Transformers

Evaluamos el sentimiento de cada titular deportivo real usando el clasificador BERT multilingüe.


In [ ]:
try:
    classifier = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")
    
    def get_sentiment_score(text):
        res = classifier(text)[0]
        stars = int(res['label'][0])
        score = (stars - 3) / 2.0  # Mapear 1-5 a rango [-1.0, 1.0]
        return score
        
    news_df['sentiment_score'] = news_df['headline'].apply(get_sentiment_score)
    print("Análisis de sentimiento completado mediante HuggingFace sobre noticias reales.")
except Exception as e:
    print(f"Error en pipeline: {e}. Aplicando análisis léxico de fallback.")
    news_df['sentiment_score'] = news_df['headline'].apply(api_helper.analyze_sentiment_lexical)

# Mostrar muestra de resultados
print(news_df.head(10))


### 3. Extracción de Entidades y Keywords en Noticias Reales (SpaCy)

Buscamos menciones a lesiones o bajas y contabilizamos las entidades reconocidas por SpaCy en las noticias reales.


In [ ]:
try:
    nlp = spacy.load("es_core_news_sm")
except:
    try:
        nlp = spacy.load("en_core_web_sm")
    except:
        nlp = spacy.blank("es")

def extract_nlp_features(text):
    doc = nlp(text)
    lower_text = text.lower()
    
    # Identificar palabras clave de lesiones o bajas
    has_injury = int(any(kw in lower_text for kw in [
        'lesión', 'lesionado', 'baja', 'molestias', 'duda', 'suspendido', 'sanción',
        'injury', 'injured', 'sidelined', 'miss', 'out'
    ]))
    entities = [ent.text for ent in doc.ents]
    return pd.Series([has_injury, len(entities)], index=['injury_flag', 'ent_count'])

nlp_feats = news_df['headline'].apply(extract_nlp_features)
news_df = pd.concat([news_df, nlp_feats], axis=1)
print("Extracción con SpaCy completada sobre noticias reales.")
print(news_df.head(10))


### 4. Configuración de Features NLP en el Dataset de Partidos Históricos

Dado que no es posible recuperar noticias históricas reales para partidos disputados en décadas pasadas, definimos las columnas de features NLP con valores por defecto (neutrales). Esto asegura que el modelo aprenda de características estructurales sólidas durante el entrenamiento, y que podamos inyectar el sentimiento y las lesiones reales en tiempo real durante las predicciones en vivo.


In [ ]:
# Rellenar con valores por defecto (neutrales) para el dataset histórico
df['sentiment_score_home'] = 0.0
df['sentiment_score_away'] = 0.0
df['injury_flag_home'] = 0
df['injury_flag_away'] = 0
df['news_volume_home'] = 0
df['news_volume_away'] = 0

output_path = os.path.join(PROCESSED_DIR, "features_nlp.csv")
df.to_csv(output_path, index=False)
print(f"Dataset de partidos guardado con éxito con columnas NLP neutrales en: {output_path}")
print(f"Dimensiones del dataset: {df.shape}")


### Limitaciones y Decisiones de Diseño Responsable (NLP)

- **Eliminación del Corpus Artificial:** En cumplimiento con las mejores prácticas de IA responsable, se eliminó la generación de noticias sintéticas basadas en plantillas repetitivas para evitar el sobreajuste a patrones artificiales y el sesgo metodológico.
- **Análisis Exclusivo de Noticias Reales:** El sistema ahora está diseñado para conectarse y analizar únicamente noticias reales. Históricamente, las features se rellenan con valores por defecto neutrales (0), mientras que el análisis real-time en el dashboard calcula la señal en vivo usando Google News RSS.
- **Privacidad y Ética:** Las búsquedas de noticias están limitadas a términos del ámbito deportivo profesional y figuras públicas (selecciones nacionales de fútbol), sin almacenar datos personales de usuarios ni información confidencial.
